# Kurzarbeitsbereitschaft

Dieses Notebook prüft rückblickend je abgeschlossenem Kalendermonat, ob die
Organisation die Voraussetzungen für Kurzarbeit erfüllt hätte: mindestens 30 % aller
Mitarbeitenden gelten als kurzarbeitsfähig, wenn ihr Anteil interner Arbeit
mindestens 24 % beträgt und ihr kumulierter Überstundenstand zum Monatsende unter
14 Stunden liegt (siehe `spec/spec-kurzarbeit.md`).

**Vollständig unabhängig von der Umsatzprognose** des Hauptdashboards
(`01_dashboard.ipynb`) und den Bausteinen Bestand, Schulungsanmeldungen und Kosten -
ein Kapazitäts-/Personalsignal, kein Umsatz- oder Kostensignal. Es liest die Daten
nur; es verändert nichts.

**Für einzelne Personen bleibt unsichtbar, an welcher Bedingung sie scheitern** - alle
Ausgaben in diesem Notebook zeigen ausschließlich Aggregatzahlen.

**So wird es benutzt:** oben im Menü *Laufzeit → Alle ausführen*, dann von oben nach
unten lesen.

In [ ]:
# @title Umgebung einrichten und Kurzarbeit-Rohdaten laden

import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py

import setup

anzahl_monate = 6

rohdaten = setup.kurzarbeit_rohdaten(anzahl_monate=anzahl_monate)

## Rollenzuordnung und Schwellenwerte

Die Rollenzuordnung (wer laut Geschäftsführung/Vertrieb nie in den Zähler
kurzarbeitsfähiger Personen eingeht) ist eine personenbezogene Angabe und steht
deshalb in keiner Datei dieses Repositories - sie wird zur Laufzeit aus der
Umgebungsvariable/dem Colab-Secret `KURZARBEIT_ROLLENZUORDNUNG` gelesen.

Die drei Schwellenwerte sind unten frei änderbar, um Was-wäre-wenn-Szenarien
("ab welcher Schwelle wäre die Organisation vorbereitet gewesen") ohne Neuabruf
durchzuspielen - ein erneutes Ausführen dieser und der folgenden Zelle reicht.

In [ ]:
# @title Rollenzuordnung und Schwellenwerte konfigurieren

from umsatzprognose.clockodo import rollenzuordnung_automatisch
from umsatzprognose.domaene import Schwellenwerte

rollenzuordnung = rollenzuordnung_automatisch()

# Standardwerte laut Spec 5.5 - hier änderbar.
schwellenwerte = Schwellenwerte(
    anteil_interne_arbeit=0.24,
    ueberstunden_stunden=14.0,
    quote_organisation=0.30,
)

## Kurzarbeitsbereitschaft je Monat

Die Gesamtzahl einbezogener Personen als größerer Balken, die tatsächlich
kurzarbeitsfähigen als schmalerer Balken davor, dazu die Quote als Linie auf einer
zweiten y-Achse - sandbraun, wenn die Schwelle erreicht wurde, sonst violett (bewusst
keine Grün/Rot-Wertung: "erreicht" ist hier kein gutes Ergebnis). Werte per
Mauszeiger über Balken/Punkten. Darunter dieselben Zahlen als Tabelle.

In [ ]:
# @title Kurzarbeitsbereitschaft je Monat

from umsatzprognose.darstellung import diagramme
from umsatzprognose.domaene import bewertungen

ergebnisse = bewertungen(rohdaten, rollenzuordnung=rollenzuordnung, schwellenwerte=schwellenwerte)
monate = sorted(ergebnisse)

diagramme.kurzarbeit_grafik(ergebnisse)

## Kurzarbeitsbereitschaft je Monat (Tabelle)

Der Standardmonat ist der jüngste abgeschlossene Monat; darunter eine Tabelle über
alle geladenen zurückliegenden Monate. Ausschließlich Aggregatzahlen - Anzahl und
Anteil kurzarbeitsfähiger Personen, die Zähler für die einzelnen Scheiter-Gründe,
ausgeschlossene und nicht bestimmbare Personen, sowie die verwendeten
Schwellenwerte.

In [ ]:
# @title Kurzarbeitsbereitschaft je Monat (Tabelle)

MONATSNAMEN = (
    "Januar",
    "Februar",
    "März",
    "April",
    "Mai",
    "Juni",
    "Juli",
    "August",
    "September",
    "Oktober",
    "November",
    "Dezember",
)

# Dieselbe (nicht wertende) Farbfamilie wie KURZARBEIT_SCHWELLE_ERREICHT/
# KURZARBEIT_SCHWELLE_NICHT_ERREICHT in darstellung/gestaltung.py (dort fuer die
# Grafik oben) - hier als ANSI-Truecolor-Code fuer denselben Unterschied im reinen
# Textoutput dieser Zelle. Jupyter stellt ANSI-Farbcodes in der Zellenausgabe dar.
_ANSI_ERFUELLT = "\033[38;2;138;109;79m"
_ANSI_NICHT_ERFUELLT = "\033[38;2;74;58;167m"
_ANSI_RESET = "\033[0m"


def status_text(bewertung) -> str:
    if bewertung.vorbereitet is None:
        return "keine Auswertung möglich (keine einbezogene Person)"
    return "Voraussetzung erfüllt" if bewertung.vorbereitet else "Voraussetzung nicht erfüllt"


def status_farbe(bewertung) -> str:
    if bewertung.vorbereitet is None:
        return ""
    return _ANSI_ERFUELLT if bewertung.vorbereitet else _ANSI_NICHT_ERFUELLT


def quote_text(bewertung) -> str:
    if bewertung.quote is None:
        return "n/a"
    return f"{bewertung.quote:.1%}"


letzter_monat = monate[-1]
letzte_bewertung = ergebnisse[letzter_monat]
jahr, monat_nr = letzter_monat

print(
    f"{MONATSNAMEN[monat_nr - 1]} {jahr}: "
    f"{status_farbe(letzte_bewertung)}{status_text(letzte_bewertung)}{_ANSI_RESET}"
)
print(
    f"  Quote kurzarbeitsfähiger Personen: {quote_text(letzte_bewertung)}"
    f" (Schwelle {schwellenwerte.quote_organisation:.0%}),"
    f" {letzte_bewertung.anzahl_kurzarbeitsfaehig} von {letzte_bewertung.anzahl_einbezogen}"
)
print(f"  Scheitert an interner Arbeit: {letzte_bewertung.anzahl_scheitert_interne_arbeit}")
print(f"  Scheitert an Überstunden:    {letzte_bewertung.anzahl_scheitert_ueberstunden}")
print(f"  Scheitert an beidem:         {letzte_bewertung.anzahl_scheitert_beide}")
print(f"  Ausgeschlossen (Rollenzuordnung): {letzte_bewertung.anzahl_ausgeschlossen}")
print(f"  Nicht bestimmbar:                 {letzte_bewertung.anzahl_nicht_bestimmbar}")
print(
    f"  Schwellenwerte: Anteil interne Arbeit >= {schwellenwerte.anteil_interne_arbeit:.0%},"
    f" Überstundenstand < {schwellenwerte.ueberstunden_stunden:.0f}h,"
    f" Quote >= {schwellenwerte.quote_organisation:.0%}"
)

print()
print(f"{'Monat':<16} {'Status':<26} {'Quote':>7} {'KA':>4} {'Ausgeschl.':>11} {'N.best.':>8}")
for monat in monate:
    bewertung = ergebnisse[monat]
    jahr, monat_nr = monat
    bezeichnung = f"{MONATSNAMEN[monat_nr - 1]} {jahr}"
    # status_text() wird zuerst auf die Zielbreite gepolstert, erst danach mit den
    # ANSI-Codes umschlossen - sonst zaehlten deren unsichtbare Zeichen mit und die
    # Spalten dahinter wuerden verrutschen.
    status_gepolstert = f"{status_text(bewertung):<26}"
    print(
        f"{bezeichnung:<16} {status_farbe(bewertung)}{status_gepolstert}{_ANSI_RESET} "
        f"{quote_text(bewertung):>7}"
        f" {bewertung.anzahl_kurzarbeitsfaehig:>4} {bewertung.anzahl_ausgeschlossen:>11}"
        f" {bewertung.anzahl_nicht_bestimmbar:>8}"
    )

## Was zu den Zahlen zu wissen ist

Ein Hinweis erscheint je Monat, wenn Personen ausgeschlossen, unklassifizierte
Stunden aufgetreten sind, oder Personen nicht bestimmbar waren - jeweils nur als
Zähler, ohne Namen oder Einzelwerte.

In [ ]:
# @title Hinweise zu den Zahlen

gab_hinweise = False
for monat in monate:
    hinweise = ergebnisse[monat].hinweise
    if not hinweise:
        continue
    gab_hinweise = True
    jahr, monat_nr = monat
    print(f"{MONATSNAMEN[monat_nr - 1]} {jahr}:")
    for hinweis in hinweise:
        print(f"  - {hinweis}")

if not gab_hinweise:
    print("Keine Hinweise.")